# rewrite_data_files MVP v2 — Simple Shortcut Implementation

This notebook demonstrates a simplified compaction approach for PyIceberg based on the maintainer's feedback:
- **Scope down to whole table**: No filters or matching, compact the entire table at once.
- **Binpack via shortcut**: Read the table into Arrow and write it right back out using the existing PyIceberg tools.
- **REPLACE operation**: Leverage `table.overwrite()` which automatically cleans up old files and adds properly sized new files.

---
## Part 1: Setup — Create a table with the small-files problem

In [2]:
import pyarrow as pa
import os, shutil, random
from collections import defaultdict

from pyiceberg.catalog.sql import SqlCatalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, StringType, LongType
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import IdentityTransform

WAREHOUSE = "/tmp/iceberg_mvp_compaction_v2"
DB_PATH = "/tmp/iceberg_mvp_compaction_v2.db"
for path in [WAREHOUSE, DB_PATH]:
    if os.path.exists(path):
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)

catalog = SqlCatalog("test", uri=f"sqlite:///{DB_PATH}", warehouse=f"file://{WAREHOUSE}")
try:
    catalog.create_namespace("db")
except:
    pass

schema = Schema(
    NestedField(1, "id", LongType()),
    NestedField(2, "name", StringType()),
    NestedField(3, "category", StringType()),
    NestedField(4, "value", LongType()),
)
spec = PartitionSpec(
    PartitionField(source_id=3, field_id=1000, transform=IdentityTransform(), name="category")
)

table = catalog.create_table("db.events_v2", schema=schema, partition_spec=spec)

# Create 18 small files (6 per partition)
categories = ["electronics", "clothing", "food"]
for i in range(18):
    table.append(pa.table({
        "id": list(range(i*100, (i+1)*100)),
        "name": [f"item_{i}_{j}" for j in range(100)],
        "category": [categories[i % 3]] * 100,
        "value": [random.randint(1, 10000) for _ in range(100)],
    }))

print(f"✅ Table created with {len(list(table.scan().plan_files()))} data files")
print(f"   {table.scan().to_arrow().num_rows} total rows across 3 partitions")


✅ Table created with 18 data files
   1800 total rows across 3 partitions


In [3]:
# Inspect the state BEFORE compaction
print("=" * 80)
print("BEFORE COMPACTION")
print("=" * 80)

before_tasks = list(table.scan().plan_files())
before_files_by_part = defaultdict(list)
for t in before_tasks:
    before_files_by_part[str(t.file.partition)].append(t)

for part, tasks in before_files_by_part.items():
    total_size = sum(t.file.file_size_in_bytes for t in tasks)
    print(f"  Partition {part}: {len(tasks)} files, {total_size:,} bytes")
    for t in tasks:
        print(f"    {os.path.basename(t.file.file_path):45s} "
              f"{t.file.file_size_in_bytes:>6,} bytes  rows={t.file.record_count}")

total_files_before = len(before_tasks)
total_size_before = sum(t.file.file_size_in_bytes for t in before_tasks)
print(f"  TOTAL: {total_files_before} files, {total_size_before:,} bytes")

BEFORE COMPACTION
  Partition Record[food]: 6 files, 14,613 bytes
    00000-0-a156c7cc-f327-44fc-adf5-750e784ed840.parquet  2,426 bytes  rows=100
    00000-0-fffe7ca6-d78f-487f-8246-9c303187553e.parquet  2,433 bytes  rows=100
    00000-0-accba82e-ae40-4b85-88f9-15b56f85264b.parquet  2,426 bytes  rows=100
    00000-0-9300d815-d650-4891-80e6-70256cb3567b.parquet  2,421 bytes  rows=100
    00000-0-ca247d20-2f0d-4239-bad8-b7f922c9d41a.parquet  2,432 bytes  rows=100
    00000-0-97a7c38f-0af2-477b-8c5f-256d2ba3e714.parquet  2,475 bytes  rows=100
  Partition Record[clothing]: 6 files, 14,735 bytes
    00000-0-759d031d-c8f8-4e91-b6cf-a4b0f8a67b37.parquet  2,453 bytes  rows=100
    00000-0-8b2657bb-0045-4cc2-aa3c-7b6907d7f93d.parquet  2,453 bytes  rows=100
    00000-0-b0d37443-5e81-4dc0-8d99-fdf0df1de9f7.parquet  2,448 bytes  rows=100
    00000-0-dc4ffb0f-8be0-427f-98ae-31f50b8f508c.parquet  2,455 bytes  rows=100
    00000-0-41a7bdda-fecb-41d9-ba6e-fff1a44b0e21.parquet  2,453 bytes  rows=100
  

---
## Part 2: The MVP Compaction `compact()`

This uses the "shortcut" strategy: we don't selectively plan individual files or groups.
Instead, we use PyIceberg's built-in target file bin-packing natively by reading the whole table to Arrow, and executing `table.overwrite()`.

In [4]:
def compact(table):
    """
    Simple compaction harness that reads the whole table and writes it back.
    Serves as the foundation for a future table.maintenance.compact() API.
    """
    print("Starting Compaction...")
    
    # 1. Read the table (pull existing data into Arrow).
    # This gives us all the data correctly typed and partitioned.
    print("  -> Reading current table state into Arrow memory (scan -> to_arrow)...")
    arrow_table = table.scan().to_arrow()
    print(f"  -> Read {arrow_table.num_rows} total rows.")

    # 2. Overwrite the table using the REPLACE operation.
    # PyIceberg's overwrite() creates new optimally sized data files 
    # (via write.target-file-size-bytes default mechanics) and removes the old ones in an atomic commit.
    print("  -> Overwriting table gracefully (atomically deleting old files & writing new scaled files)...")
    table.overwrite(arrow_table)
    
    print("✅ Compaction complete!")

# Run compaction
compact(table)

Starting Compaction...
  -> Reading current table state into Arrow memory (scan -> to_arrow)...
  -> Read 1800 total rows.
  -> Overwriting table gracefully (atomically deleting old files & writing new scaled files)...
✅ Compaction complete!


---
## Part 3: Verification — Prove it worked

In [5]:
table.refresh()

after_tasks = list(table.scan().plan_files())
after_files_by_part = defaultdict(list)
for t in after_tasks:
    after_files_by_part[str(t.file.partition)].append(t)

print("=" * 80)
print("AFTER COMPACTION")
print("=" * 80)

for part, tasks in after_files_by_part.items():
    total_size = sum(t.file.file_size_in_bytes for t in tasks)
    print(f"  Partition {part}: {len(tasks)} files, {total_size:,} bytes")
    for t in tasks:
        print(f"    {os.path.basename(t.file.file_path):45s} "
              f"{t.file.file_size_in_bytes:>6,} bytes  rows={t.file.record_count}")

total_files_after = len(after_tasks)
print(f"  TOTAL FILES: {total_files_before} ➔ {total_files_after} (reduced by {total_files_before - total_files_after})")
assert total_files_after < total_files_before, "File count should have decreased!"
assert table.scan().to_arrow().num_rows == 1800, "Row counts should remain identical!"
print("✅ ALL CHECKS PASSED!")

AFTER COMPACTION
  Partition Record[food]: 1 files, 6,204 bytes
    00000-0-44f173ad-7df5-4d96-93ec-7bdeadfbd13d.parquet  6,204 bytes  rows=600
  Partition Record[electronics]: 1 files, 6,426 bytes
    00000-1-44f173ad-7df5-4d96-93ec-7bdeadfbd13d.parquet  6,426 bytes  rows=600
  Partition Record[clothing]: 1 files, 6,254 bytes
    00000-2-44f173ad-7df5-4d96-93ec-7bdeadfbd13d.parquet  6,254 bytes  rows=600
  TOTAL FILES: 18 ➔ 3 (reduced by 15)
✅ ALL CHECKS PASSED!


---
## Part 4: Using the Native Implementation
Here we create a fresh table and demonstrate the native `table.maintenance.compact()` we added to `pyiceberg.table.maintenance`.


In [6]:
# Setup a new table to test the actual integrated Implementation
print("=" * 80)
print("TESTING NATIVE API: table.maintenance.compact()")
print("=" * 80)

# Create a fresh database and table
native_db = "/tmp/iceberg_mvp_compaction_native.db"
native_wh = "/tmp/iceberg_mvp_compaction_native"
for path in [native_db, native_wh]:
    if os.path.exists(path):
        import shutil
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)

native_catalog = SqlCatalog("native_test", uri=f"sqlite:///{native_db}", warehouse=f"file://{native_wh}")
native_catalog.create_namespace("native_db")
native_table = native_catalog.create_table("native_db.events", schema=schema, partition_spec=spec)

# Populate the table with small files
categories = ["electronics", "clothing", "food"]
for i in range(18):
    native_table.append(pa.table({
        "id": list(range(i*100, (i+1)*100)),
        "name": [f"native_item_{i}_{j}" for j in range(100)],
        "category": [categories[i % 3]] * 100,
        "value": [random.randint(1, 10000) for _ in range(100)],
    }))

before_count = len(list(native_table.scan().plan_files()))
print(f"\n✅ Native Table created with {before_count} data files")
print(f"   {native_table.scan().to_arrow().num_rows} total rows")

print("\nStarting Native Compaction via new table.maintenance.compact()...")
native_table.maintenance.compact()
print("✅ Native Compaction complete!")

native_table.refresh()
after_count = len(list(native_table.scan().plan_files()))
print(f"\nRESULTS:")
print(f"  TOTAL FILES: {before_count} ➔ {after_count} (reduced by {before_count - after_count})")
assert after_count < before_count, "Native implementation did not reduce file count!"
assert native_table.scan().to_arrow().num_rows == 1800, "Native implementation lost rows!"
print("\n✅ Native Verification Passed!")

TESTING NATIVE API: table.maintenance.compact()

✅ Native Table created with 18 data files
   1800 total rows

Starting Native Compaction via new table.maintenance.compact()...
✅ Native Compaction complete!

RESULTS:
  TOTAL FILES: 18 ➔ 3 (reduced by 15)

✅ Native Verification Passed!
